# Opdracht 4 – ETL Implementatie Data Warehouse
## Robbert & Mees

In deze opdracht implementeren wij de ETL-processen van het Data Warehouse. 
We laden data vanuit het Source Data Model (SDM) naar het DWH en passen 
Slowly Changing Dimensions Type 1 en Type 2 toe.

## Inlaadstrategie

In deze opdracht maken wij gebruik van de **Incremental Data Loading** strategie.

Reden:
- we willen alleen nieuwe en gewijzigde data verwerken
- nodig voor SCD Type 2 (historie behouden)


Setup & Connecties (NOG INVULLEN)

In [13]:
import pandas as pd
import sqlite3
from loguru import logger
from datetime import datetime

sdm_conn = sqlite3.connect('../Source Data Model/BikeToDrive_V3_OLTP.db', timeout=30)
dwh_conn = sqlite3.connect('../Data Warehouse/BikeToDrive_DWH.db', timeout=30)
sdm_cursor = sdm_conn.cursor()
dwh_cursor = dwh_conn.cursor()

logger.info("SDM en DWH connecties succesvol.")

2026-03-29 14:12:32.201 | INFO     | __main__:<module>:11 - SDM en DWH connecties succesvol.


Dim_Klant (SCD TYPE 2) ROBBERT

In [18]:
# Wat willen we bereiken?
# We willen Dim_Klant in het DWH vullen met deze logica:
# - nieuwe klant in SDM, nog niet in DWH -> INSERT
# - bestaande klant, maar gewijzigd -> oude actuele rij afsluiten + nieuwe INSERT
# - bestaande klant, niet gewijzigd -> niets doen.

# business key = klantnr uit SDM
# surrogate key = klant_sk in DWH

# We halen eerst brondata op uit beide klanttabellen.
# Dit doen we omdat Dim_Klant niet uit één bron komt, maar uit de tabellen:
# - Accessoire_Verkoop_Klant
# - Fiets_Verkoop_Klant
# We voegen deze dus daarom ook verticaal samen.

query_accessoire_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Accessoire_Verkoop_Klant
"""

query_fiets_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Fiets_Verkoop_Klant
"""

df_accessoire_klant = pd.read_sql_query(query_accessoire_klant, sdm_conn)
df_fiets_klant = pd.read_sql_query(query_fiets_klant, sdm_conn)

# We willen deze twee losse DataFrames verticaal samenvoegen.
# We maken dus één bronset voor Dim_Klant.
df_klant_source = pd.concat(
    [df_accessoire_klant, df_fiets_klant],
    ignore_index=True
)

# We verwijderen dubbele klanten op basis van de business key klantnr.
df_klant_source = df_klant_source.drop_duplicates(subset=["klantnr"]).reset_index(drop=True)

# Nu halen we de actuele klantrecords op uit het DWH.
# Bij SCD Type 2 willen we namelijk alleen vergelijken met de actuele rij van een klant,
# niet met de oude historische versies.
# We herkennen de actuele regel aan: eindtijd IS NULL.

query_dwh_klant = """
SELECT
    klant_sk,
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum,
    begintijd,
    eindtijd
FROM Dim_Klant
WHERE eindtijd IS NULL
"""

df_klant_dwh = pd.read_sql_query(query_dwh_klant, dwh_conn)

# Nu gaan we bepalen of een bestaande klant veranderd is.
# We vergelijken alleen inhoudelijke klantgegevens.
# We vergelijken dus NIET op:
# - klant_sk
# - begintijd
# - eindtijd
# want die horen bij de DWH-historie en niet bij de business-inhoud van de klant.

def klant_is_changed(source_row, dwh_row):
    return (
        str(source_row["naam"]) != str(dwh_row["naam"]) or
        str(source_row["adres"]) != str(dwh_row["adres"]) or
        str(source_row["woonplaats"]) != str(dwh_row["woonplaats"]) or
        str(source_row["geslacht"]) != str(dwh_row["geslacht"]) or
        str(source_row["geboortedatum"]) != str(dwh_row["geboortedatum"])
    )

# Eerst bepalen hoeveel klanten nieuw, gewijzigd of ongewijzigd zijn.
new_count = 0
changed_count = 0
preview_unchanged_count = 0

for _, src_row in df_klant_source.iterrows():
    klantnr = src_row["klantnr"]
    current_match = df_klant_dwh[df_klant_dwh["klantnr"] == klantnr]

    if current_match.empty:
        new_count += 1
    else:
        dwh_row = current_match.iloc[0]
        if klant_is_changed(src_row, dwh_row):
            changed_count += 1
        else:
            preview_unchanged_count += 1

# We gaan nu de echte ETL uitvoeren voor Dim_Klant.
now_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

inserted_count = 0
updated_count = 0
unchanged_count = 0

logger.info("Start ETL voor Dim_Klant (SCD Type 2)")

for _, src_row in df_klant_source.iterrows():
    klantnr = src_row["klantnr"]
    current_match = df_klant_dwh[df_klant_dwh["klantnr"] == klantnr]

    # 1. Nieuwe klant -> INSERT
    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Klant (
                klantnr,
                naam,
                adres,
                woonplaats,
                geslacht,
                geboortedatum,
                begintijd,
                eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, NULL)
        """, (
            int(src_row["klantnr"]),
            src_row["naam"],
            src_row["adres"],
            src_row["woonplaats"],
            src_row["geslacht"],
            src_row["geboortedatum"],
            now_ts
        ))

        inserted_count += 1

    # 2. Bestaande klant -> check of gewijzigd
    else:
        dwh_row = current_match.iloc[0]

        if klant_is_changed(src_row, dwh_row):
            # Oude actuele rij afsluiten
            dwh_conn.execute("""
                UPDATE Dim_Klant
                SET eindtijd = ?
                WHERE klant_sk = ?
            """, (
                now_ts,
                int(dwh_row["klant_sk"])
            ))

            # Nieuwe actuele versie toevoegen
            dwh_conn.execute("""
                INSERT INTO Dim_Klant (
                    klantnr,
                    naam,
                    adres,
                    woonplaats,
                    geslacht,
                    geboortedatum,
                    begintijd,
                    eindtijd
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, NULL)
            """, (
                int(src_row["klantnr"]),
                src_row["naam"],
                src_row["adres"],
                src_row["woonplaats"],
                src_row["geslacht"],
                src_row["geboortedatum"],
                now_ts
            ))

            updated_count += 1

        else:
            unchanged_count += 1

dwh_conn.commit()

logger.info(
    f"Dim_Klant klaar | inserted={inserted_count}, "
    f"updated_scd2={updated_count}, unchanged={unchanged_count}"
)

result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Klant
    ORDER BY klantnr, klant_sk
""", dwh_conn)

print(result_df)

2026-03-29 14:46:24.524 | INFO     | __main__:<module>:115 - Start ETL voor Dim_Klant (SCD Type 2)
2026-03-29 14:46:24.532 | INFO     | __main__:<module>:192 - Dim_Klant klaar | inserted=0, updated_scd2=0, unchanged=25


    klant_sk  klantnr               naam                 adres  \
0          1        1         Jan Jansen         Kerkstraat 12   
1          2        2     Sophie de Boer           Lindelaan 8   
2          3        3      Pieter Visser         Havenstraat 3   
3          4        4          Emma Smit          Boomgaard 22   
4          5        5         Tom Bakker        Stationsweg 44   
5          6        6        Lisa Meijer         Dijkstraat 10   
6          7        7      Bart de Vries      Brouwersgracht 7   
7          8        8     Julia van Dijk         Plataanlaan 5   
8          9        9          Kevin Mol             Singel 99   
9         10       10         Nina Groen        Waterstraat 16   
10        11       11       Daan Willems           Parklaan 31   
11        12       12            Eva Vos            Zomerweg 2   
12        13       13        Rik de Jong      Noorderstraat 88   
13        14       14       Mila Kuipers    Heemraadssingel 15   
14        

Dim_Accessoire (SCD TYPE 1) ROBBERT

In [19]:
# Wat willen we bereiken?
# We willen Dim_Accessoire in het DWH vullen met SCD Type 1 logica:
# - nieuw accessoire in SDM, nog niet in DWH -> INSERT
# - bestaand accessoire, maar gewijzigd -> UPDATE (overschrijven)
# - bestaand accessoire, niet gewijzigd -> niets doen

# business key = accessoirenr uit SDM
# surrogate key = accessoire_sk in DWH

# 1. Brondata ophalen per bron (horizontale samenvoeging)
# Voor Dim_Accessoire komt de data uit:
# - Accessoire_Verkoop_Accessoire + Accessoire_Verkoop_Leverancier
# - Accessoire_Inkoop_Accessoire + Accessoire_Inkoop_Leverancier
#
# We JOINEN eerst per bron, omdat accessoire- en leveranciersdata
# in aparte tabellen staan.

query_accessoire_verkoop = """
SELECT
    a.accessoirenr,
    a.naam,
    a.soort,
    l.leveranciernr,
    l.naam AS leveranciernaam,
    l.adres AS leverancieradres,
    l.woonplaats AS leverancierplaats
FROM Accessoire_Verkoop_Accessoire a
JOIN Accessoire_Verkoop_Leverancier l
    ON a.leverancier = l.leveranciernr
"""

query_accessoire_inkoop = """
SELECT
    a.accessoirenr,
    a.naam,
    a.soort,
    l.leveranciernr,
    l.naam AS leveranciernaam,
    l.adres AS leverancieradres,
    l.woonplaats AS leverancierplaats
FROM Accessoire_Inkoop_Accessoire a
JOIN Accessoire_Inkoop_Leverancier l
    ON a.leverancier = l.leveranciernr
"""

df_accessoire_verkoop = pd.read_sql_query(query_accessoire_verkoop, sdm_conn)
df_accessoire_inkoop = pd.read_sql_query(query_accessoire_inkoop, sdm_conn)

logger.info(f"Aantal accessoire-rijen uit verkoopbron: {len(df_accessoire_verkoop)}")
logger.info(f"Aantal accessoire-rijen uit inkoopbron: {len(df_accessoire_inkoop)}")

# 2. Verticale samenvoeging tot één bronset
# We zetten de rijen uit verkoop en inkoop onder elkaar.
# Daarna houden we per business key nog maar één record over.

df_accessoire_source = pd.concat(
    [df_accessoire_verkoop, df_accessoire_inkoop],
    ignore_index=True
)

df_accessoire_source = df_accessoire_source.drop_duplicates(subset=["accessoirenr"]).reset_index(drop=True)

logger.info(f"Aantal unieke accessoires in source: {len(df_accessoire_source)}")

# 3. Actuele records uit het DWH ophalen
# We vergelijken met de huidige inhoud van Dim_Accessoire.

query_dwh_accessoire = """
SELECT
    accessoire_sk,
    accessoirenr,
    naam,
    soort,
    leveranciernr,
    leveranciernaam,
    leverancieradres,
    leverancierplaats,
    begintijd,
    eindtijd
FROM Dim_Accessoire
WHERE eindtijd IS NULL
"""

df_accessoire_dwh = pd.read_sql_query(query_dwh_accessoire, dwh_conn)

logger.info(f"Aantal actuele accessoires in DWH: {len(df_accessoire_dwh)}")

# 4. Helperfunctie: is het accessoire veranderd?
# Bij SCD Type 1 vergelijken we eerst of de inhoudelijke attributen veranderd zijn.
# Alleen dan voeren we een UPDATE uit.

def accessoire_is_changed(source_row, dwh_row):
    return (
        str(source_row["naam"]) != str(dwh_row["naam"]) or
        str(source_row["soort"]) != str(dwh_row["soort"]) or
        str(source_row["leveranciernr"]) != str(dwh_row["leveranciernr"]) or
        str(source_row["leveranciernaam"]) != str(dwh_row["leveranciernaam"]) or
        str(source_row["leverancieradres"]) != str(dwh_row["leverancieradres"]) or
        str(source_row["leverancierplaats"]) != str(dwh_row["leverancierplaats"])
    )

# 5. Vooraf bepalen hoeveel records nieuw / gewijzigd / ongewijzigd zijn
new_count = 0
changed_count = 0
preview_unchanged_count = 0

for _, src_row in df_accessoire_source.iterrows():
    accessoirenr = src_row["accessoirenr"]

    current_match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == accessoirenr]

    if current_match.empty:
        new_count += 1
    else:
        dwh_row = current_match.iloc[0]

        if accessoire_is_changed(src_row, dwh_row):
            changed_count += 1
        else:
            preview_unchanged_count += 1

logger.info(f"Aantal nieuwe accessoires: {new_count}")
logger.info(f"Aantal gewijzigde accessoires: {changed_count}")
logger.info(f"Aantal ongewijzigde accessoires: {preview_unchanged_count}")

# 6. Echte ETL uitvoeren met SCD Type 1
# - nieuw -> INSERT
# - gewijzigd -> UPDATE
# - ongewijzigd -> niets doen

now_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

inserted_count = 0
updated_count = 0
unchanged_count = 0

logger.info("Start ETL voor Dim_Accessoire (SCD Type 1)")

for _, src_row in df_accessoire_source.iterrows():
    accessoirenr = src_row["accessoirenr"]

    current_match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == accessoirenr]

    # Nieuw accessoire -> INSERT
    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Accessoire (
                accessoirenr,
                naam,
                soort,
                leveranciernr,
                leveranciernaam,
                leverancieradres,
                leverancierplaats,
                begintijd,
                eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (
            int(src_row["accessoirenr"]),
            src_row["naam"],
            src_row["soort"],
            int(src_row["leveranciernr"]),
            src_row["leveranciernaam"],
            src_row["leverancieradres"],
            src_row["leverancierplaats"],
            now_ts
        ))

        inserted_count += 1

    # Bestaand accessoire -> check of gewijzigd
    else:
        dwh_row = current_match.iloc[0]

        if accessoire_is_changed(src_row, dwh_row):
            # Bij SCD Type 1 overschrijven we de bestaande rij.
            dwh_conn.execute("""
                UPDATE Dim_Accessoire
                SET
                    naam = ?,
                    soort = ?,
                    leveranciernr = ?,
                    leveranciernaam = ?,
                    leverancieradres = ?,
                    leverancierplaats = ?
                WHERE accessoire_sk = ?
            """, (
                src_row["naam"],
                src_row["soort"],
                int(src_row["leveranciernr"]),
                src_row["leveranciernaam"],
                src_row["leverancieradres"],
                src_row["leverancierplaats"],
                int(dwh_row["accessoire_sk"])
            ))

            updated_count += 1

        else:
            unchanged_count += 1

dwh_conn.commit()

logger.info(
    f"Dim_Accessoire klaar | inserted={inserted_count}, "
    f"updated_scd1={updated_count}, unchanged={unchanged_count}"
)

# 7. Controle
result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Accessoire
    ORDER BY accessoirenr, accessoire_sk
""", dwh_conn)

print(result_df)

2026-03-29 15:30:48.194 | INFO     | __main__:<module>:49 - Aantal accessoire-rijen uit verkoopbron: 10
2026-03-29 15:30:48.195 | INFO     | __main__:<module>:50 - Aantal accessoire-rijen uit inkoopbron: 13
2026-03-29 15:30:48.201 | INFO     | __main__:<module>:63 - Aantal unieke accessoires in source: 13
2026-03-29 15:30:48.213 | INFO     | __main__:<module>:86 - Aantal actuele accessoires in DWH: 0
2026-03-29 15:30:48.223 | INFO     | __main__:<module>:122 - Aantal nieuwe accessoires: 13
2026-03-29 15:30:48.224 | INFO     | __main__:<module>:123 - Aantal gewijzigde accessoires: 0
2026-03-29 15:30:48.225 | INFO     | __main__:<module>:124 - Aantal ongewijzigde accessoires: 0
2026-03-29 15:30:48.226 | INFO     | __main__:<module>:137 - Start ETL voor Dim_Accessoire (SCD Type 1)
2026-03-29 15:30:48.245 | INFO     | __main__:<module>:205 - Dim_Accessoire klaar | inserted=13, updated_scd1=0, unchanged=0


    accessoire_sk  accessoirenr                      naam        soort  \
0               1             1              LED voorlamp  Verlichting   
1               2             2           LED achterlicht  Verlichting   
2               3             3  USB-oplaadbare fietslamp  Verlichting   
3               4             4              Ringslot AXA       Sloten   
4               5             5    Ketting met cijferslot       Sloten   
5               6             6         U-slot met beugel       Sloten   
6               7             7    Dubbele fietstas zwart       Tassen   
7               8             8   Enkele schouderfietstas       Tassen   
8               9             9      Waterdichte zadeltas       Tassen   
9              10            10      Kinderfietsmand roze       Tassen   
10             11            11  Reflecterende spaakclips  Verlichting   
11             12            12        Vouwbaar kabelslot       Sloten   
12             13            13       

Dim_Datum ROBBERT

In [20]:
# Wat willen we bereiken?
# We willen Dim_Datum in het DWH vullen.
# Voor Dim_Datum geldt:
# - nieuwe datum in SDM, nog niet in DWH -> INSERT
# - bestaande datum in DWH -> niets doen
#
# Voor Dim_Datum gebruiken we geen SCD Type 1 of Type 2,
# omdat datumwaarden zelf niet wijzigen.

# business key = datum
# surrogate key = datum_sk in DWH

# 1. Brondata ophalen uit de relevante feitbronnen=
# We halen de datums op uit:
# - Fiets_Verkoop
# - Accessoire_Verkoop
# - Onderhoud
#
# Voor Fct_Inkoop hebben we in het SDM geen echte datumkolom,
# maar maand + jaar. Daarom gebruiken we hier alleen de tabellen
# die een volledige datum bevatten.

query_fiets_verkoop_datum = """
SELECT datum
FROM Fiets_Verkoop
"""

query_accessoire_verkoop_datum = """
SELECT datum
FROM Accessoire_Verkoop
"""

query_onderhoud_datum = """
SELECT datum
FROM Onderhoud
"""

df_fiets_verkoop_datum = pd.read_sql_query(query_fiets_verkoop_datum, sdm_conn)
df_accessoire_verkoop_datum = pd.read_sql_query(query_accessoire_verkoop_datum, sdm_conn)
df_onderhoud_datum = pd.read_sql_query(query_onderhoud_datum, sdm_conn)

logger.info(f"Aantal datum-rijen uit Fiets_Verkoop: {len(df_fiets_verkoop_datum)}")
logger.info(f"Aantal datum-rijen uit Accessoire_Verkoop: {len(df_accessoire_verkoop_datum)}")
logger.info(f"Aantal datum-rijen uit Onderhoud: {len(df_onderhoud_datum)}")

# 2. Verticale samenvoeging tot één bronset
# We zetten alle datumrijen onder elkaar.
# Daarna houden we alleen unieke datums over.

df_datum_source = pd.concat(
    [df_fiets_verkoop_datum, df_accessoire_verkoop_datum, df_onderhoud_datum],
    ignore_index=True
)

df_datum_source = df_datum_source.drop_duplicates(subset=["datum"]).reset_index(drop=True)

logger.info(f"Aantal unieke datums in source: {len(df_datum_source)}")

# 3. Datumonderdelen afleiden
# We zetten de datumkolom om naar datetime,
# zodat we year, month en day kunnen afleiden.

df_datum_source["datum"] = pd.to_datetime(df_datum_source["datum"])
df_datum_source["year"] = df_datum_source["datum"].dt.year
df_datum_source["month"] = df_datum_source["datum"].dt.month
df_datum_source["day"] = df_datum_source["datum"].dt.day

# Voor opslag in SQLite zetten we datum weer om naar YYYY-MM-DD string
df_datum_source["datum"] = df_datum_source["datum"].dt.strftime("%Y-%m-%d")

# 4. Bestaande datums uit DWH ophalen
query_dwh_datum = """
SELECT
    datum_sk,
    datum,
    year,
    month,
    day
FROM Dim_Datum
"""

df_datum_dwh = pd.read_sql_query(query_dwh_datum, dwh_conn)

logger.info(f"Aantal datums in DWH: {len(df_datum_dwh)}")

# 5. Bepalen welke datums nieuw zijn

new_count = 0
existing_count = 0

for _, src_row in df_datum_source.iterrows():
    datum = src_row["datum"]

    current_match = df_datum_dwh[df_datum_dwh["datum"] == datum]

    if current_match.empty:
        new_count += 1
    else:
        existing_count += 1

logger.info(f"Aantal nieuwe datums: {new_count}")
logger.info(f"Aantal bestaande datums: {existing_count}")

# 6. Echte ETL uitvoeren
# Alleen nieuwe datums worden toegevoegd.

inserted_count = 0

logger.info("Start ETL voor Dim_Datum")

for _, src_row in df_datum_source.iterrows():
    datum = src_row["datum"]

    current_match = df_datum_dwh[df_datum_dwh["datum"] == datum]

    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Datum (
                datum,
                year,
                month,
                day
            )
            VALUES (?, ?, ?, ?)
        """, (
            src_row["datum"],
            int(src_row["year"]),
            int(src_row["month"]),
            int(src_row["day"])
        ))

        inserted_count += 1

dwh_conn.commit()

logger.info(f"Dim_Datum klaar | inserted={inserted_count}")


# 7. Controle
result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Datum
    ORDER BY datum
""", dwh_conn)

print(result_df)

2026-03-29 15:47:05.141 | INFO     | __main__:<module>:42 - Aantal datum-rijen uit Fiets_Verkoop: 150
2026-03-29 15:47:05.143 | INFO     | __main__:<module>:43 - Aantal datum-rijen uit Accessoire_Verkoop: 100
2026-03-29 15:47:05.145 | INFO     | __main__:<module>:44 - Aantal datum-rijen uit Onderhoud: 50
2026-03-29 15:47:05.155 | INFO     | __main__:<module>:57 - Aantal unieke datums in source: 196
2026-03-29 15:47:05.224 | INFO     | __main__:<module>:84 - Aantal datums in DWH: 0
2026-03-29 15:47:05.276 | INFO     | __main__:<module>:101 - Aantal nieuwe datums: 196
2026-03-29 15:47:05.277 | INFO     | __main__:<module>:102 - Aantal bestaande datums: 0
2026-03-29 15:47:05.279 | INFO     | __main__:<module>:109 - Start ETL voor Dim_Datum
2026-03-29 15:47:05.339 | INFO     | __main__:<module>:136 - Dim_Datum klaar | inserted=196


     datum_sk       datum  year  month  day
0           6  2024-01-02  2024      1    2
1         123  2024-01-06  2024      1    6
2          20  2024-01-08  2024      1    8
3         168  2024-01-10  2024      1   10
4          85  2024-01-11  2024      1   11
..        ...         ...   ...    ...  ...
191        61  2024-12-26  2024     12   26
192       183  2024-12-27  2024     12   27
193       172  2024-12-28  2024     12   28
194        95  2024-12-30  2024     12   30
195       119  2024-12-31  2024     12   31

[196 rows x 5 columns]


Fct_Verkoop ROBBERT


In [ ]:
# Wat willen we bereiken?
# We willen Fct_Verkoop vullen met verkoopgebeurtenissen uit:
# - Fiets_Verkoop
# - Accessoire_Verkoop
#
# In het DWH komt dit samen in één feittabel:
# Fct_Verkoop
#
# De feittabel bevat:
# - verkoopnr
# - datum_sk
# - aantal
# - totaalprijs
# - standaardprijs
# - klant_sk
# - fiets_sk of accessoire_sk
# - monteur_sk
#
# Hiervoor moeten we de business keys uit het SDM omzetten naar surrogate keys uit het DWH.

# 1. Brondata ophalen uit beide verkoopbronnen
# We JOINEN hier ook meteen met de producttabellen om standaardprijs op te halen,
# omdat standaardprijs een meetwaarde is die in de feittabel hoort.

query_fiets_verkoop = """
SELECT
    fv.fiets_verkoopnr AS verkoopnr,
    fv.datum,
    fv.aantal,
    fv.verkoopprijs AS totaalprijs,
    ff.standaardprijs,
    fv.klant,
    fv.fiets,
    fv.monteur,
    NULL AS accessoire
FROM Fiets_Verkoop fv
JOIN Fiets_Verkoop_Fiets ff
    ON fv.fiets = ff.fietsnr
"""

query_accessoire_verkoop = """
SELECT
    av.accessoire_verkoopnr AS verkoopnr,
    av.datum,
    av.aantal,
    av.verkoopprijs AS totaalprijs,
    aa.standaardprijs,
    av.klant,
    NULL AS fiets,
    av.monteur,
    av.accessoire
FROM Accessoire_Verkoop av
JOIN Accessoire_Verkoop_Accessoire aa
    ON av.accessoire = aa.accessoirenr
"""

df_fiets_verkoop = pd.read_sql_query(query_fiets_verkoop, sdm_conn)
df_accessoire_verkoop = pd.read_sql_query(query_accessoire_verkoop, sdm_conn)

logger.info(f"Aantal verkooprijen uit Fiets_Verkoop: {len(df_fiets_verkoop)}")
logger.info(f"Aantal verkooprijen uit Accessoire_Verkoop: {len(df_accessoire_verkoop)}")

# 2. Controle op overlap in verkoopnummers
# Omdat Fct_Verkoop één primaire sleutel verkoopnr heeft,
# controleren we of fiets- en accessoireverkoop toevallig hetzelfde verkoopnr gebruiken.

overlap_verkoopnrs = set(df_fiets_verkoop["verkoopnr"]).intersection(
    set(df_accessoire_verkoop["verkoopnr"])
)

if overlap_verkoopnrs:
    raise ValueError(
        f"Er zijn overlappende verkoopnummers tussen Fiets_Verkoop en "
        f"Accessoire_Verkoop: {sorted(overlap_verkoopnrs)}. "
        f"Met het huidige DWH-schema kan Fct_Verkoop deze niet allebei opslaan."
    )

# 3. Verticale samenvoeging tot één bronset
df_verkoop_source = pd.concat(
    [df_fiets_verkoop, df_accessoire_verkoop],
    ignore_index=True
)

logger.info(f"Totaal aantal verkooprijen in bronset: {len(df_verkoop_source)}")

# 4. Actuele dimensies uit het DWH ophalen
# Feiten moeten verwijzen naar de actuele surrogate keys van de dimensies.

df_datum_dwh = pd.read_sql_query("""
SELECT
    datum_sk,
    datum
FROM Dim_Datum
""", dwh_conn)

df_klant_dwh = pd.read_sql_query("""
SELECT
    klant_sk,
    klantnr
FROM Dim_Klant
WHERE eindtijd IS NULL
""", dwh_conn)

df_fiets_dwh = pd.read_sql_query("""
SELECT
    fiets_sk,
    fietsnr
FROM Dim_Fiets
WHERE eindtijd IS NULL
""", dwh_conn)

df_accessoire_dwh = pd.read_sql_query("""
SELECT
    accessoire_sk,
    accessoirenr
FROM Dim_Accessoire
WHERE eindtijd IS NULL
""", dwh_conn)

df_monteur_dwh = pd.read_sql_query("""
SELECT
    monteur_sk,
    monteurnr
FROM Dim_Monteur
WHERE eindtijd IS NULL
""", dwh_conn)

# 5. Bestaande verkooprecords uit DWH ophalen
# Zo detecteren we welke verkooprijen nog niet voorkomen in het DWH.

df_verkoop_dwh = pd.read_sql_query("""
SELECT
    verkoopnr
FROM Fct_Verkoop
""", dwh_conn)

logger.info(f"Aantal bestaande verkooprijen in DWH: {len(df_verkoop_dwh)}")

# 6. Helperfuncties voor surrogate key lookups

def get_datum_sk(datum_value):
    match = df_datum_dwh[df_datum_dwh["datum"] == str(datum_value)]
    if match.empty:
        raise ValueError(f"Geen datum_sk gevonden voor datum: {datum_value}")
    return int(match.iloc[0]["datum_sk"])

def get_klant_sk(klantnr):
    match = df_klant_dwh[df_klant_dwh["klantnr"] == int(klantnr)]
    if match.empty:
        raise ValueError(f"Geen klant_sk gevonden voor klantnr: {klantnr}")
    return int(match.iloc[0]["klant_sk"])

def get_fiets_sk(fietsnr):
    match = df_fiets_dwh[df_fiets_dwh["fietsnr"] == int(fietsnr)]
    if match.empty:
        raise ValueError(f"Geen fiets_sk gevonden voor fietsnr: {fietsnr}")
    return int(match.iloc[0]["fiets_sk"])

def get_accessoire_sk(accessoirenr):
    match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == int(accessoirenr)]
    if match.empty:
        raise ValueError(f"Geen accessoire_sk gevonden voor accessoirenr: {accessoirenr}")
    return int(match.iloc[0]["accessoire_sk"])

def get_monteur_sk(monteurnr):
    match = df_monteur_dwh[df_monteur_dwh["monteurnr"] == int(monteurnr)]
    if match.empty:
        raise ValueError(f"Geen monteur_sk gevonden voor monteurnr: {monteurnr}")
    return int(match.iloc[0]["monteur_sk"])


# 7. Vooraf bepalen hoeveel verkooprijen nieuw zijn
new_count = 0
existing_count = 0

for _, src_row in df_verkoop_source.iterrows():
    verkoopnr = src_row["verkoopnr"]
    current_match = df_verkoop_dwh[df_verkoop_dwh["verkoopnr"] == verkoopnr]

    if current_match.empty:
        new_count += 1
    else:
        existing_count += 1

logger.info(f"Aantal nieuwe verkooprijen: {new_count}")
logger.info(f"Aantal bestaande verkooprijen: {existing_count}")

# 8. Echte ETL uitvoeren voor Fct_Verkoop
# Omdat dit een feittabel is:
# - nieuwe verkoop -> INSERT
# - bestaande verkoop -> niets doen
# We gaan ervan uit dat bestaande feiten niet overschreven worden.

inserted_count = 0
skipped_count = 0

logger.info("Start ETL voor Fct_Verkoop")

for _, src_row in df_verkoop_source.iterrows():
    verkoopnr = src_row["verkoopnr"]

    current_match = df_verkoop_dwh[df_verkoop_dwh["verkoopnr"] == verkoopnr]

    # Alleen nieuwe feiten invoegen
    if current_match.empty:
        datum_sk = get_datum_sk(src_row["datum"])
        klant_sk = get_klant_sk(src_row["klant"])
        monteur_sk = get_monteur_sk(src_row["monteur"])

        # Business rule:
        # of fiets, of accessoire
        if pd.notna(src_row["fiets"]):
            fiets_sk = get_fiets_sk(src_row["fiets"])
            accessoire_sk = None
        else:
            fiets_sk = None
            accessoire_sk = get_accessoire_sk(src_row["accessoire"])

        dwh_conn.execute("""
            INSERT INTO Fct_Verkoop (
                verkoopnr,
                datum_sk,
                aantal,
                totaalprijs,
                standaardprijs,
                klant_sk,
                fiets_sk,
                accessoire_sk,
                monteur_sk
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            int(src_row["verkoopnr"]),
            datum_sk,
            int(src_row["aantal"]),
            float(src_row["totaalprijs"]),
            float(src_row["standaardprijs"]),
            klant_sk,
            fiets_sk,
            accessoire_sk,
            monteur_sk
        ))

        inserted_count += 1
    else:
        skipped_count += 1

dwh_conn.commit()

logger.info(
    f"Fct_Verkoop klaar | inserted={inserted_count}, skipped_existing={skipped_count}"
)

# 9. Controle
result_df = pd.read_sql_query("""
    SELECT *
    FROM Fct_Verkoop
    ORDER BY verkoopnr
""", dwh_conn)

print(result_df)

Dim_Fiets (SCD Type 2) MEES

Dim_Monteur (SCD Type 1) MEES

Fct_Onderhoud MEES

Fct_Inkoop MEES